# Summarize a raw news article with the trained model

Pipeline, the same one the training data went through:

    data/raw_news/{name}.txt ──clean_text──> cleaned article ──BPE──> <s> article </s>
      ──TransformerEncoder (prefix-LM), greedy, one token at a time──> summary </s>

Run this with the project's `.venv` kernel on a machine with a CUDA GPU (the model uses CuPy).
The model needs a few hundred MB of GPU memory; if `train-model` is still running on the same
GPU, wait for it to finish or you may run either process out of memory.

In [ ]:
# 1. Imports. The notebook sits in the project root, so `src` is importable from here.
import re
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").is_dir():
    raise RuntimeError(f"Open this notebook from the project root, not {PROJECT_ROOT}")
sys.path.insert(0, str(PROJECT_ROOT))

from src.config import ProjectConfig
from src.data_preprocessing.batching import SUMMARY_SUFFIX, CreateTrainingBatch
from src.data_preprocessing.text_cleaning_pipeline import clean_text, contains_english, text_to_unicode
from src.inference.greedy_decoder import greedy_summarize
from src.models.encoder import TransformerEncoder
from src.training.train import load_checkpoint

In [ ]:
# 2. Tokenizer + model with the same settings as training, then load the trained weights.
config = ProjectConfig.from_yaml(PROJECT_ROOT / "config" / "config.yaml")
RAW_DIR = PROJECT_ROOT / "data" / "raw_news"
SUMMARY_DIR = config.resolve(config.summary_dir)  # mT5 reference summaries, for comparison

# Epoch 8 is the final model (epoch 9 was lost). Set to None to pick the newest summarizer_epoch_N.npz.
CHECKPOINT = config.resolve(config.checkpoint_dir) / "summarizer_epoch_8.npz"
if CHECKPOINT is None:
    # End-of-epoch files only (summarizer_epoch_N.npz), not mid-epoch _step_ or .bak files.
    checkpoints = sorted(
        (p for p in config.resolve(config.checkpoint_dir).glob("summarizer_epoch_*.npz")
         if re.fullmatch(r"summarizer_epoch_\d+", p.stem)),
        key=lambda p: int(p.stem.rsplit("_", 1)[1]),
    )
    if not checkpoints:
        raise FileNotFoundError("No summarizer_epoch_*.npz in checkpoints/; train the model first")
    CHECKPOINT = checkpoints[-1]

vocab_dir = config.resolve(config.vocab_dir)
batcher = CreateTrainingBatch(  # only its tokenizer, special-token IDs and length budgets are used
    batch_size=1,
    vocab_json_data=vocab_dir / "vocab.json",
    merges_txt_data=vocab_dir / "merges.txt",
    max_article_length=config.max_article_length,
    max_summary_length=config.max_summary_length,
)
model = TransformerEncoder(
    vocab_size=batcher.vocab_size,
    d_model=config.d_model,
    num_heads=config.num_heads,
    num_layers=config.num_layers,
    d_ff=config.d_ff,
    max_seq_length=config.max_sequence_length,
    tie_weights=config.tie_weights,
)
load_checkpoint(model, CHECKPOINT)
print(f"Loaded {Path(CHECKPOINT).name} | V={batcher.vocab_size} D={config.d_model} N={config.num_layers}")

In [ ]:
# 3. Clean a raw article exactly like scripts/clean_data.py, then summarize it.
def clean_article(path: Path) -> str:
    """Raw file -> cleaned one-line article (UTF-8 fix, NFC, strip HTML, drop ZWJ/ZWNJ, collapse whitespace)."""
    raw_text = text_to_unicode(path)
    if contains_english(raw_text):
        # clean_data.py skips these files, so the model never trained on text like this.
        print(f"Warning: {path.name} contains English letters; training data excluded such articles.")
    cleaned = clean_text(raw_text)
    if not cleaned:
        raise ValueError(f"{path.name} is empty after cleaning")
    return cleaned


def summarize_file(path: str | Path) -> str:
    path = Path(path)
    article = clean_article(path)

    article_tokens = len(batcher.tokenizer.encode(article).ids)
    budget = config.max_article_length - 2  # <s> and </s>
    if article_tokens > budget:
        print(f"Note: article has {article_tokens} tokens; only the first {budget} reach the model.")
    return greedy_summarize(model, batcher, article)

In [ ]:
# 4. Pick any file from data/raw_news and summarize it.
ARTICLE = RAW_DIR / "100001.txt"

summary = summarize_file(ARTICLE)

print("Article:\n", clean_article(ARTICLE)[:1000], "...\n")
print("Model summary:\n", summary, "\n")

reference = SUMMARY_DIR / f"{ARTICLE.stem}{SUMMARY_SUFFIX}"
if reference.is_file():
    print("mT5 reference summary (training target):\n", reference.read_text(encoding="utf-8"))

In [ ]:
# 5. Or summarize text you paste in directly (it goes through the same cleaning).
TEXT = """
यहाँ नेपाली समाचार टाँस्नुहोस् ।
"""

print(greedy_summarize(model, batcher, clean_text(TEXT)))